<a href="https://colab.research.google.com/github/chiraswykj/ML2_Lab_ChiraswyKJ_4NI23CI028/blob/main/ML_Exp5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math

# ============================================================
# FOIL - First Order Inductive Learner
# ============================================================

# ------------------------------------------------------------
# Example dataset
# Target predicate: PlayTennis
# ------------------------------------------------------------

examples = [
    # Outlook, Temperature, Humidity, Wind, Target
    ("Sunny",    "Hot",  "High",   "Weak",   True),
    ("Sunny",    "Hot",  "High",   "Strong", False),
    ("Overcast", "Hot",  "High",   "Weak",   True),
    ("Rain",     "Mild", "High",   "Weak",   True),
    ("Rain",     "Cool", "Normal", "Weak",   True),
    ("Rain",     "Cool", "Normal", "Strong", False),
    ("Overcast", "Cool", "Normal", "Strong", True),
    ("Sunny",    "Mild", "High",   "Weak",   False),
    ("Sunny",    "Cool", "Normal", "Weak",   True),
    ("Rain",     "Mild", "Normal", "Weak",   True),
    ("Sunny",    "Mild", "Normal", "Strong", True),
    ("Overcast", "Mild", "High",   "Strong", True),
    ("Overcast", "Hot",  "Normal", "Weak",   True),
    ("Rain",     "Mild", "High",   "Strong", False)
]

attributes = [
    "Outlook",
    "Temperature",
    "Humidity",
    "Wind"
]

# Convert examples into dictionaries
data = []

for row in examples:
    example = {
        "Outlook": row[0],
        "Temperature": row[1],
        "Humidity": row[2],
        "Wind": row[3],
        "Target": row[4]
    }
    data.append(example)


# ------------------------------------------------------------
# Separate Positive and Negative examples
# ------------------------------------------------------------

Pos = [x for x in data if x["Target"] == True]
Neg = [x for x in data if x["Target"] == False]

print("Positive Examples:", len(Pos))
print("Negative Examples:", len(Neg))


# ------------------------------------------------------------
# Check whether an example satisfies a rule
# ------------------------------------------------------------

def satisfies(example, rule):
    """
    rule = list of conditions
    Example:
    [('Outlook', 'Sunny'), ('Humidity', 'High')]
    """

    for attribute, value in rule:
        if example[attribute] != value:
            return False

    return True


# ------------------------------------------------------------
# Get examples covered by a rule
# ------------------------------------------------------------

def covered_examples(examples, rule):
    return [x for x in examples if satisfies(x, rule)]


# ------------------------------------------------------------
# FOIL Gain
# ------------------------------------------------------------

def foil_gain(rule, literal, Pos, Neg):

    # Examples covered before adding literal
    old_pos = covered_examples(Pos, rule)
    old_neg = covered_examples(Neg, rule)

    p0 = len(old_pos)
    n0 = len(old_neg)

    # Add the new literal
    new_rule = rule + [literal]

    # Examples covered after adding literal
    new_pos = covered_examples(Pos, new_rule)
    new_neg = covered_examples(Neg, new_rule)

    p1 = len(new_pos)
    n1 = len(new_neg)

    # If no positive examples remain, gain is 0
    if p1 == 0:
        return 0

    # If old rule covers no positive examples
    if p0 == 0:
        return 0

    # If the new rule covers no negative examples,
    # give it a very high gain
    if n1 == 0:
        return float("inf")

    # FOIL Gain formula
    gain = p1 * (
        math.log2(p1 / (p1 + n1))
        -
        math.log2(p0 / (p0 + n0))
    )

    return gain


# ------------------------------------------------------------
# Generate candidate literals
# ------------------------------------------------------------

def generate_candidate_literals(rule, Pos, Neg):

    candidates = []

    used_attributes = [attribute for attribute, value in rule]

    for attribute in attributes:

        # Don't use the same attribute twice
        if attribute in used_attributes:
            continue

        values = set()

        for example in Pos + Neg:
            values.add(example[attribute])

        for value in values:
            candidates.append((attribute, value))

    return candidates


# ------------------------------------------------------------
# Learn ONE new rule
# ------------------------------------------------------------

def learn_new_rule(Pos, Neg):

    # NewRule initially predicts True with no conditions
    NewRule = []

    print("\nStarting a new rule:")
    print("Target =", "True")

    while True:

        covered_neg = covered_examples(Neg, NewRule)

        # If no negative examples are covered,
        # rule is complete
        if len(covered_neg) == 0:
            break

        # Generate candidate literals
        candidates = generate_candidate_literals(
            NewRule, Pos, Neg
        )

        if not candidates:
            break

        # Find literal with maximum FOIL Gain
        best_literal = None
        best_gain = -float("inf")

        for literal in candidates:

            gain = foil_gain(
                NewRule,
                literal,
                Pos,
                Neg
            )

            print(
                "Candidate:",
                literal,
                "Gain:",
                round(gain, 4) if gain != float("inf") else "INF"
            )

            if gain > best_gain:
                best_gain = gain
                best_literal = literal

        # Add best literal
        NewRule.append(best_literal)

        print(
            "Selected:",
            best_literal,
            "Gain:",
            round(best_gain, 4)
            if best_gain != float("inf")
            else "INF"
        )

    return NewRule


# ------------------------------------------------------------
# Main FOIL Algorithm
# ------------------------------------------------------------

def FOIL(Pos, Neg):

    Learned_rules = []

    Pos = Pos.copy()

    while Pos:

        print("\n====================================")
        print("Remaining Positive Examples:", len(Pos))
        print("====================================")

        # Learn a new rule
        NewRule = learn_new_rule(Pos, Neg)

        # Add rule to learned rules
        Learned_rules.append(NewRule)

        print("\nNew Rule Learned:")
        print(NewRule)

        # Remove positive examples covered by new rule
        covered_pos = covered_examples(Pos, NewRule)

        Pos = [
            example
            for example in Pos
            if example not in covered_pos
        ]

    return Learned_rules


# ------------------------------------------------------------
# Run FOIL
# ------------------------------------------------------------

Learned_rules = FOIL(Pos, Neg)


# ------------------------------------------------------------
# Display final rules
# ------------------------------------------------------------

print("\n\n====================================")
print("FINAL LEARNED RULES")
print("====================================")

for i, rule in enumerate(Learned_rules, 1):

    if len(rule) == 0:
        condition = "TRUE"
    else:
        condition = " AND ".join(
            attribute + " = " + value
            for attribute, value in rule
        )

    print(
        f"Rule {i}: PlayTennis = True IF {condition}"
    )

Positive Examples: 10
Negative Examples: 4

Remaining Positive Examples: 10

Starting a new rule:
Target = True
Candidate: ('Outlook', 'Sunny') Gain: -0.7546
Candidate: ('Outlook', 'Overcast') Gain: INF
Candidate: ('Outlook', 'Rain') Gain: -0.7546
Candidate: ('Temperature', 'Hot') Gain: 0.2112
Candidate: ('Temperature', 'Mild') Gain: -0.3981
Candidate: ('Temperature', 'Cool') Gain: 0.2112
Candidate: ('Humidity', 'Normal') Gain: 1.5782
Candidate: ('Humidity', 'High') Gain: -1.2877
Candidate: ('Wind', 'Strong') Gain: -1.5437
Candidate: ('Wind', 'Weak') Gain: 2.0495
Selected: ('Outlook', 'Overcast') Gain: INF

New Rule Learned:
[('Outlook', 'Overcast')]

Remaining Positive Examples: 6

Starting a new rule:
Target = True
Candidate: ('Outlook', 'Sunny') Gain: 0.0
Candidate: ('Outlook', 'Rain') Gain: 0.0
Candidate: ('Temperature', 'Hot') Gain: -0.263
Candidate: ('Temperature', 'Mild') Gain: 0.0
Candidate: ('Temperature', 'Cool') Gain: 0.304
Candidate: ('Humidity', 'Normal') Gain: 1.6601
Cand

In [ ]:
def covers(rule, example):
    """
    Checks whether all conditions in the rule are satisfied
    by an example.
    """
    for condition in rule:
        if condition not in example:
            return False
    return True


def foil(examples, target):
    """
    Basic FOIL algorithm.

    examples : list of sets
        Each example contains facts/features.
    target : target class/fact
    """

    positive = []
    negative = []

    # Separate positive and negative examples
    for example in examples:
        if target in example:
            positive.append(example)
        else:
            negative.append(example)

    rules = []

    while positive:

        rule = []

        # Specialization of the rule
        while True:

            # If rule covers no negative examples, rule is complete
            covered_negative = [
                ex for ex in negative
                if covers(rule, ex)
            ]

            if not covered_negative:
                break

            # Find possible conditions
            candidates = set()

            for ex in positive:
                for item in ex:
                    if item != target and item not in rule:
                        candidates.add(item)

            if not candidates:
                break

            # Select the condition that removes the most negatives
            best_condition = None
            best_score = -1

            for condition in candidates:

                score = 0

                for ex in covered_negative:
                    if condition not in ex:
                        score += 1

                if score > best_score:
                    best_score = score
                    best_condition = condition

            if best_condition is None:
                break

            rule.append(best_condition)

        rules.append(rule)

        # Remove positive examples covered by this rule
        positive = [
            ex for ex in positive
            if not covers(rule, ex)
        ]

    return rules


# ---------------------------------------------------
# Example Dataset
# ---------------------------------------------------

examples = [
    {"bird", "flies", "has_wings"},
    {"bird", "flies", "feathers"},
    {"bird", "cannot_fly", "penguin"},
    {"animal", "cannot_fly"},
    {"bird", "flies", "small"},
]

target = "flies"

rules = foil(examples, target)

print("Learned FOIL Rules:")
for i, rule in enumerate(rules, 1):
    print(f"Rule {i}: IF {' AND
   '.join(rule)} THEN {target}")


Learned FOIL Rules:
Rule 1: IF small THEN flies
Rule 2: IF has_wings THEN flies
Rule 3: IF feathers THEN flies
